In [ ]:
import gffutils
import pyfaidx
from Bio.Seq import Seq
import subprocess
from Bio import SeqIO

from Bio import Phylo

import pandas as pd
import json


In [3]:
#prep work, split filtered fasta into pgcoe and rv, align both to reference, remove reference from aligned output

#for target in ['RSVA', 'RSVB']:
for target in []:
    fasta = f'inputs/{target}_filtered.fasta'
    pgcoerecs = []
    rvrecs = []
    for record in SeqIO.parse(fasta, "fasta"):
        record.id = record.id
        if("Yale-RSV" in record.id):
            record.id = record.id.replace("_", "-")
            pgcoerecs.append(record)
        elif("Yale-RV" in record.id):
            record.id = record.id.split("_")[0]
            rvrecs.append(record)
    SeqIO.write(pgcoerecs, fasta.replace("_filtered.fasta", "_pgcoe.fasta"), "fasta")
    SeqIO.write(rvrecs, fasta.replace("_filtered.fasta", "_rv.fasta"), "fasta")

    for out in ['pgcoe', 'rv']:
        gmafftf = fasta.replace("_filtered.fasta", f"_{out}_aligned.fasta")
        # Align sequences to reference
        with open(gmafftf, "w") as f: 
            subprocess.run(['mafft', '--auto', '--thread', '4', '--keeplength', '--add',
                            fasta.replace("_filtered.fasta", f"_{out}.fasta"), f'refs/{target}.fasta'], 
                            stdout=f, text=True)

        aligned_records = list(SeqIO.parse(gmafftf, "fasta"))
        filtered_records = aligned_records[1:] # Exclude the reference sequence (first record)
        # Overwrite gmafftf with filtered records (without the original reference)
        with open(gmafftf, "w") as out_handle:
            SeqIO.write(filtered_records, out_handle, "fasta")


In [4]:
def extract_gene_sequences(fasta_path, gff_path, gene_name, nuc_out, aa_out):
    """
    Extract nucleotide and amino acid sequences for a specific gene from a multi-sequence FASTA and GFF3.
    Args:
        fasta_path (str): Path to input FASTA file (aligned or unaligned).
        gff_path (str): Path to GFF3 file.
        gene_name (str): Name of the gene to extract.
        nuc_out (str): Output path for nucleotide sequences.
        aa_out (str): Output path for amino acid sequences.
    """
    db = gffutils.create_db(gff_path, dbfn=':memory:')
    fasta = pyfaidx.Fasta(fasta_path)
    # Find the CDS feature for the gene
    cds_list = [cds for cds in db.features_of_type('CDS', order_by='start') if cds.attributes.get('gene', [''])[0] == gene_name]
    if not cds_list:
        raise ValueError(f"Gene {gene_name} not found in GFF3.")
    cds = cds_list[0]
    seqid = cds.seqid
    start = cds.start - 1  # pyfaidx uses 0-based indexing
    end = cds.end

    # Clear output files
    open(aa_out, "w").close()
    open(nuc_out, "w").close()

    with open(aa_out, "a") as faa, open(nuc_out, "a") as fnuc:
        for seq_record in list(fasta.keys()):
            nstops = [Seq(fasta[seq_record][(start+n):(end+n)].seq.replace('-', 'N')).translate().count('*') for n in [0,1,2]]
            n = nstops.index(min(nstops))
            nseq = fasta[seq_record][(start+n):(end+n)].seq
            aa = Seq(nseq.replace('-', 'N')).translate()
            faa.write(f">{seq_record}    {gene_name} (n={n})\n{aa}\n")
            fnuc.write(f">{seq_record}    {gene_name}\n{nseq.replace('-', 'N')}\n")



def filter_n_sequences(nuc_fasta, aa_fasta, nuc_out, aa_out, n_threshold=0.2):
    """
    Remove sequences from nucleotide and amino acid FASTA files if the nucleotide sequence has > n_threshold fraction of 'n'.
    Args:
        nuc_fasta (str): Path to nucleotide FASTA file.
        aa_fasta (str): Path to amino acid FASTA file.
        nuc_out (str): Output path for filtered nucleotide FASTA.
        aa_out (str): Output path for filtered amino acid FASTA.
        n_threshold (float): Fraction threshold for 'n' bases (default 0.2).
    """
    nuc_records = SeqIO.to_dict(SeqIO.parse(nuc_fasta, "fasta"))
    aa_records = SeqIO.to_dict(SeqIO.parse(aa_fasta, "fasta"))

    keep_ids = []
    drop_ids = []
    for rec_id, rec in nuc_records.items():
        seq = str(rec.seq).lower()
        n_frac = seq.count('n') / len(seq) if len(seq) > 0 else 1.0
        if n_frac <= n_threshold:
            keep_ids.append(rec_id)
        else:
            drop_ids.append(rec_id)

    with open(nuc_out, "w") as nf, open(aa_out, "w") as af:
        SeqIO.write([nuc_records[i] for i in keep_ids if i in nuc_records], nf, "fasta")
        SeqIO.write([aa_records[i] for i in keep_ids if i in aa_records], af, "fasta")
    return(len(keep_ids), len(drop_ids))

In [5]:
def extract_reference_gene_aa(fasta_path, gff_path, prefix=""):
    """
    Extract reference amino acid sequences for each CDS from a reference fasta and gff,
    and write all to a single output file.
    Args:
        fasta_path (str): Path to reference FASTA file.
        gff_path (str): Path to GFF3 file.
        output_dir (str): Output directory for amino acid sequences.
    """
    ref_fasta = pyfaidx.Fasta(fasta_path)
    db = gffutils.create_db(gff_path, dbfn=':memory:')
    for cds in db.features_of_type('CDS', order_by='start'):
        gene = cds.attributes['gene'][0]
        with open(f"{prefix}{gene}_aa.fasta", "w") as f:
            seqid = cds.seqid
            start = cds.start - 1  # pyfaidx uses 0-based
            end = cds.end
            ref_seq = ref_fasta[seqid][start:end]
            nstops = [Seq(ref_seq[(n):(end-start+n)].seq.replace('-', 'N')).translate().count('*') for n in [0,1,2]]
            n = nstops.index(min(nstops))
            nseq = ref_seq[(n):(end-start+n)].seq
            aa = Seq(nseq.replace('-', 'N')).translate()
            f.write(f">{seqid}    {gene} (n={n})\n{aa}\n")


In [6]:
# # get sequences for gene region for all consensus sequences
# # filter sequences with > 20% Ns

# for cds in db.features_of_type('CDS', order_by='start'):
#     gene = cds.attributes['gene'][0]
#     extract_gene_sequences(fastaf, gfff, gene, f"geneseqs/{gene}_sequences.fasta", f"geneseqs/{target}_{gene}_aa.fasta")

#     nkeep, ndrop = filter_n_sequences(f"geneseqs/{gene}_sequences.fasta", f"geneseqs/{target}_{gene}_aa.fasta", 
#                        f"geneseqs/{gene}_sequences_filt.fasta", f"geneseqs/{target}_{gene}_aa_filt.fasta", 
#                         n_threshold=0.2)
#     print(f"{gene:5s} kept {nkeep:4d}, dropped{ndrop:4d}  (pass:{nkeep / (nkeep + ndrop):6.2%})")


In [7]:
def run_iqtree(iqprefix, codonf):
    """Run IQ-TREE on the codon alignment.""" 
    subprocess.run(["iqtree", "--prefix", iqprefix, "-s", codonf, 
                "-m", "GTR+G", "-nt", "AUTO", "-bb", "1000", "-alrt", "1000",
                "--redo"], check=True)


In [8]:
def get_codon_alignment(gaafastaf, gfastaf, reffasta, gmafftf, codonf):
    """Generate a codon alignment using pal2nal from an amino acid alignment and the corresponding nucleotide sequences."""
    pal2nal = "/opt/miniconda3/envs/hyphy/bin/pal2nal.pl"

    #run mafft to align aa sequences
    command = ["mafft", '--keeplength',
               #'--localpair', '--maxiterate', '1000', '--op', '3.0',
               '--auto', '--op', '3.0',
               '--anysymbol', '--add', gaafastaf, reffasta]
    print(" ".join(command))
    with open(gmafftf, 'w') as outfile, open(gmafftf + ".err", 'w') as errfile:
        subprocess.run(command, stdout=outfile, stderr=errfile, check=True)

    # Parse MAFFT output and remove the original reference sequence from reffasta
    ref_id = list(SeqIO.parse(reffasta, "fasta"))[0].id
    aligned_records = list(SeqIO.parse(gmafftf, "fasta"))
    filtered_records = [rec for rec in aligned_records if rec.id != ref_id]
    # Overwrite gmafftf with filtered records (without the original reference)
    with open(gmafftf, "w") as out_handle:
        SeqIO.write(filtered_records, out_handle, "fasta")


    # Run pal2nal to generate codon alignment from the MAFFT output and the nucleotide fasta
    command = [pal2nal, gmafftf, gfastaf, "-nomismatch", "-nogap", "-output", "fasta"]
    print(" ".join(command))
    with open(codonf, "w") as outfile:
        subprocess.run(command, stdout=outfile, check=True)

def trim_stop_codons(codonf, codontrimf, mincodons=10):
    """Trim codon alignment to remove sequences after the first stop codon found in any sequence."""
    # Define stop codons
    STOP_CODONS = {"TAA", "TAG", "TGA"}

    records = list(SeqIO.parse(codonf, "fasta"))
    L = len(records[0].seq)

    n_codons = L // 3
    trim_at = n_codons  # default: keep full length
    checkcodons = min(mincodons, n_codons)  # check up to last 'mincodons'(10) codons

    # Look for stops in the last 'mincodons' codons
    for rec in records:
        codons = [str(rec.seq[i:i+3]).upper() for i in range(0, L, 3)]
        for offset in range(checkcodons, 0, -1):  # check from -checkcodons to -1 codons
            idx = n_codons - offset
            if idx < 0:
                continue
            if codons[idx] in STOP_CODONS:
                trim_at = min(trim_at, idx)  # earliest stop position
                break

    # Convert codon index back to nt length
    trim_nt = trim_at * 3
    print(f"Trimming alignment {L//3} -> {trim_at} codons ({trim_nt} nt).")

    with open(codontrimf, "w") as out:
        for rec in records:
            rec.seq = rec.seq[:trim_nt]
            SeqIO.write(rec, out, "fasta")


In [20]:
def remove_zero_length_branches(tree):
    """
    Removes clades with a branch_length of 0 from a Biopython Tree object.
    This effectively prunes zero-length branches and creates polytomies.
    """
    # Create a list to store clades to be processed (children of zero-length clades)
    clades_to_process = []

    # Iterate through all clades in the tree
    for clade in tree.find_clades():
        # Check if the clade has children and a branch_length of 0
        if clade.branch_length == 0 and clade.clades:
            clades_to_process.append(clade)
    print(f"Removing {len(clades_to_process)} zero-length branches")
    # Process clades with zero-length branches
    for zero_clade in clades_to_process:
        parent = tree.get_path(zero_clade)[-2]  # Get the parent of the zero-length clade
        
        if parent:
            # Remove the zero-length clade from its parent's children
            parent.clades.remove(zero_clade)
            # Add the children of the zero-length clade directly to the parent
            parent.clades.extend(zero_clade.clades)

    return tree

def prune_tree_to_seqs(wgstree, fastaf, pruned_tree_file):
    """
    Prune a tree to only include tips present in the fasta file.
    Args:
        wgstree (str): Path to the input tree file (Newick format).
        fastaf (str): Path to the fasta file containing sequence names to keep.
        pruned_tree_file (str): Path to write the pruned tree.
    """
    seq_names = set(SeqIO.to_dict(SeqIO.parse(fastaf, "fasta")).keys())
    
    # Read the tree
    tree = Phylo.read(wgstree, "newick")
    tips = [c.name for c in tree.get_terminals()]
    # Find tips to prune (not in fasta)
    tips_to_prune = [term for term in tips if term not in seq_names]
    
    # Prune tips
    if(len(tips_to_prune) == len(tips)):
        print(len(tips), len(seq_names), 0)
        exit(1)
    else:
        for tip in tips_to_prune:
            tree.prune(tip)    
        #remove zero-length branches
        nozerotree = remove_zero_length_branches(tree) 

        # Write pruned tree
        print(len(tips), len(seq_names), len(tree.get_terminals()))

        Phylo.write(nozerotree, pruned_tree_file, "newick")
    return(len(tree.get_terminals()))

def filter_seqs_to_tree(fastaf, treefile, output_fasta):
    """
    Filter sequences in a FASTA file to only those present as tips in a tree file.
    Args:
        fastaf (str): Path to input FASTA file.
        treefile (str): Path to tree file (Newick format).
        output_fasta (str): Path to output filtered FASTA file.
    """
    # Get tip names from tree
    tree = Phylo.read(treefile, "newick")
    tip_names = set([term.name for term in tree.get_terminals()])
    # Parse FASTA and filter
    records = [rec for rec in SeqIO.parse(fastaf, "fasta")]
    nrecords = len(records)
    goodrecords = [rec for rec in records if rec.id in tip_names]
    # Write filtered FASTA
    with open(output_fasta, "w") as out:
        SeqIO.write(goodrecords, out, "fasta")

    print(nrecords, len(tip_names), len(goodrecords))
    return(len(goodrecords))

In [21]:
def prune_tree_to_samples(wgstree, samples, pruned_tree_file):
    """
    Prune a tree to only include tips present in the samples file.
    Args:
        wgstree (str): Path to the input tree file (Newick format).
        samples (str): Path to the samples file (one sample name per line).
        pruned_tree_file (str): Path to write the pruned tree.
    """
    with open(samples) as f:
        sample_names = set(line.strip() for line in f if line.strip())
    
    # Read the tree
    tree = Phylo.read(wgstree, "newick")
    tips = [c.name for c in tree.get_terminals()]
    # Find tips to prune (not in samples)
    tips_to_prune = [term for term in tips if term not in sample_names]
    
    # Prune tips
    if(len(tips_to_prune) == len(tips)):
        print(len(tips), len(sample_names), 0)
        exit(1)
    else:
        for tip in tips_to_prune:
            tree.prune(tip)    
        #remove zero-length branches
        nozerotree = remove_zero_length_branches(tree) 

        # Write pruned tree
        print(len(tips), len(sample_names), len(tree.get_terminals()))
        Phylo.write(nozerotree, pruned_tree_file, "newick")
    return(len(tree.get_terminals()))

def filter_seqs_to_samples(fastaf, samples, output_fasta):
    """
    Filter sequences in a FASTA file to only those present in the samples file.
    Args:
        fastaf (str): Path to input FASTA file.
        samples (str): Path to the samples file (one sample name per line).
        output_fasta (str): Path to output filtered FASTA file.
    """
    with open(samples) as f:
        sample_names = set(line.strip() for line in f if line.strip())
    
    # Parse FASTA and filter
    records = [rec for rec in SeqIO.parse(fastaf, "fasta")]
    nrecords = len(records)
    goodrecords = [rec for rec in records if rec.id in sample_names]
    # Write filtered FASTA
    with open(output_fasta, "w") as out:
        SeqIO.write(goodrecords, out, "fasta")

    print(nrecords, len(sample_names), len(goodrecords))
    return(len(goodrecords))

In [22]:
# Run FUBAR using HyPhy on the codon alignment and IQ-TREE treefile
def run_fubar(codontrimf, treefile, fubarout):
    errfile = fubarout + ".err"    
    outfile = fubarout + ".out"    
    with open(fubarout, "w") as errfile, open(outfile, "w") as outfile:
        subprocess.run([
            "hyphy", "fubar",
            "--alignment", codontrimf,
            "--tree", treefile,
            "--output", fubarout
        ], check=True,stderr=errfile, stdout=outfile)


def run_absrel(codontrimf, treefile, absrelout, branches=None):
    """Run aBSREL using HyPhy on the codon alignment and IQ-TREE treefile."""
    if branches is None:
        command = [
            "hyphy", "absrel",
            "--alignment", codontrimf,
            "--tree", treefile,
            "--output", absrelout
        ]
        print(" ".join(command))
        subprocess.run(command, check=True)
    else:
        command = [
            "hyphy", "absrel",
            "--alignment", codontrimf,
            "--tree", treefile,
            "--branches", branches,
            "--output", absrelout]
        print(" ".join(command))
        subprocess.run(command, check=True)

In [23]:
def parse_fubar_table(json_file):
    """
    Reads a fubar output file and parses out the table as a pandas DataFrame.
    Args:
        json_file (str): Path to the JSON file.
    Returns:
        pd.DataFrame: Parsed table as DataFrame.
    """
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    jsontable = data["MLE"]["content"]["0"]
    df = pd.DataFrame(jsontable)
    headers = [h[0] for h in data["MLE"]["headers"]]
    #df.columns = headers
    if(len(df.columns) > len(headers)):
        headers[len(headers):len(df.columns)] = ["x" + str(i) for i in range(len(headers), len(df.columns))]
    df.columns = headers
    df.reset_index(names='codon', inplace=True)
    df['dataset'] = json_file.replace(".json","").split("/")[-1]
    return df


In [13]:
#annotate tree with clade assignments

def annotate_tree(treefile,cladesfile):
    tree = Phylo.read(treefile, "newick")

    
    # Find a specific clade (e.g., the common ancestor of C and D)
    clade_to_highlight = tree.common_ancestor("C", "D")

    # Set the color and width of the branch leading to this clade
    clade_to_highlight.color = "red"
    clade_to_highlight.width = 3

    # Draw the tree (e.g., to ASCII or using Matplotlib)
    Phylo.draw_ascii(tree)

In [14]:
#mafft  --auto --keeplength --add inputs/RSVA_all_aligned.fasta  refs/RSVA.fasta > inputs/RSVA_all_refaligned.fasta
#mafft  --auto --keeplength --add inputs/RSVB_all_aligned.fasta  refs/RSVB.fasta > inputs/RSVB_all_refaligned.fasta
#mafft  --auto --keeplength --add inputs/RSVA_filtered.fasta  refs/RSVA.fasta > inputs/RSVA_yaleonly_aligned.fasta
#mafft  --auto --keeplength --add inputs/RSVB_filtered.fasta  refs/RSVB.fasta > inputs/RSVB_yaleonly_aligned.fasta

targets = ["RSVB", "RSVA"]
for target in targets:
    print(f"Processing {target}")
    reff = f'refs/{target}.fasta'
    gfff = f'refs/{target}.gff3'
    extract_reference_gene_aa(reff, gfff, f'./refgenes/{target}_')

db = gffutils.create_db(gfff, dbfn=':memory:')
geneids = [cds.attributes['gene'][0] for cds in db.features_of_type('CDS', order_by='start')]




Processing RSVB
Processing RSVA


/opt/miniconda3/envs/hyphy/lib/python3.13/site-packages/Bio/Seq.py:2879: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [ ]:
#PARSE CODON ALIGNMENTS FOR ALL SAMPLES

#genelist = ["G", "F"]
genelist = geneids
targets = ["RSVB", "RSVA"]
for target in targets:
    reff = f'refs/{target}.fasta'
    gfff = f'refs/{target}.gff3'
    db = gffutils.create_db(gfff, dbfn=':memory:')

    fastaf = f'inputs/{target}_all_aligned.fasta'  # <-ns aligned output + bgr 
    
    print(f"Processing {target}")
    for gene in genelist:
        reffasta = f"refgenes/{target}_{gene}_aa.fasta"                #reference amino acid sequence

        #input sequences
        gfasta = f"geneseqs/{target}_{gene}_sequences.fasta"           #nucleotide sequences
        gaafasta = f"geneseqs/{target}_{gene}_aa.fasta"                #amino acid sequences

        #make codon alignment
        gmafftf = f"geneseqs/{target}_{gene}_aa_mafft.fasta"            #amino acid alignment
        codonf = f"geneseqs/{target}_{gene}_codonaln.fasta"             #nucleotide codon alignment

        #filter tree / sequences to match
        wgstrim = f"geneseqs/{target}_{gene}_tree_pruned.nwk"           #pruned wg tree

        #hyphy outputs
        fubarout = f"hyphy/{target}_{gene}_fubar.json"                  #FUBAR output


        #extract gene region for all samples, filter sequences with > 5% Ns
        extract_gene_sequences(fastaf, gfff, gene, gfasta, gaafasta)
        nkeep, ndrop = filter_n_sequences(gfasta, gaafasta, gfasta, gaafasta, n_threshold=0.05)
        print(f"{gene:5s} kept {nkeep:4d}, dropped{ndrop:4d}  (pass:{nkeep / (nkeep + ndrop):6.2%})")


        #get codon alignment, trim to last stop codon
        get_codon_alignment(gaafasta, gfasta, reffasta, gmafftf, codonf)
        trim_stop_codons(codonf, codonf, mincodons=20)


Processing RSVB
NS1   kept 1788, dropped 100  (pass:94.70%)
mafft --keeplength --auto --op 3.0 --anysymbol --add geneseqs/RSVB_NS1_aa.fasta refgenes/RSVB_NS1_aa.fasta
/opt/miniconda3/envs/hyphy/bin/pal2nal.pl geneseqs/RSVB_NS1_aa_mafft.fasta geneseqs/RSVB_NS1_sequences.fasta -nomismatch -nogap -output fasta
Trimming alignment 130 -> 130 codons (390 nt).
NS2   kept 1817, dropped  71  (pass:96.24%)
mafft --keeplength --auto --op 3.0 --anysymbol --add geneseqs/RSVB_NS2_aa.fasta refgenes/RSVB_NS2_aa.fasta
/opt/miniconda3/envs/hyphy/bin/pal2nal.pl geneseqs/RSVB_NS2_aa_mafft.fasta geneseqs/RSVB_NS2_sequences.fasta -nomismatch -nogap -output fasta
Trimming alignment 121 -> 121 codons (363 nt).
N     kept 1845, dropped  43  (pass:97.72%)
mafft --keeplength --auto --op 3.0 --anysymbol --add geneseqs/RSVB_N_aa.fasta refgenes/RSVB_N_aa.fasta
/opt/miniconda3/envs/hyphy/bin/pal2nal.pl geneseqs/RSVB_N_aa_mafft.fasta geneseqs/RSVB_N_sequences.fasta -nomismatch -nogap -output fasta
Trimming alignment 

In [34]:
#RUN FUBAR FOR ALL SAMPLES

targets = ["RSVA", "RSVB"]
genelist = geneids
# targets = []
# genelist = []

for target in targets:
    reff = f'refs/{target}.fasta'
    gfff = f'refs/{target}.gff3'
    db = gffutils.create_db(gfff, dbfn=':memory:')
    #run fubar for all genes in genelist
    for gene in genelist:
        print(f"Processing {target} {gene}")
        #codon alignment
        codonf = f"geneseqs/{target}_{gene}_codonaln.fasta"             #nucleotide codon alignment

        #filter tree / sequences to match
        wgstree = f'inputs/{target}_tree.nwk'
        wgstrim = f"geneseqs/{target}_{gene}_tree_pruned.nwk"           #pruned wg tree

        #hyphy outputs
        fubarout = f"hyphy/{target}_{gene}_fubar.json"                  #FUBAR output
        fubercsv = f"hyphy/{target}_{gene}_fubar_table.csv"                  #FUBAR table output
        
        #prune tree to sequences, filter sequences to tree
        prune_tree_to_seqs(wgstree, codonf, wgstrim)
        n = filter_seqs_to_tree(codonf, wgstree, codonf)
        # if n > 5:
        #     run_fubar(codonf, wgstrim, fubarout)
        #     parse_fubar_table(fubarout).to_csv(fubercsv, index=False)
        # else:
        #     print(f"Skipping {target} {gene}, only {n} sequences")

Processing RSVA NS1
Removing 0 zero-length branches
1936 1724 1692
1724 1936 1692
Processing RSVA NS2
Removing 0 zero-length branches
1936 1713 1682
1713 1936 1682
Processing RSVA N
Removing 0 zero-length branches
1936 1433 1404
1433 1936 1404
Processing RSVA P
Removing 0 zero-length branches
1936 1802 1767
1802 1936 1767
Processing RSVA M
Removing 0 zero-length branches
1936 1934 1899
1934 1936 1899
Processing RSVA SH
Removing 0 zero-length branches
1936 1809 1778
1809 1936 1778
Processing RSVA G
Removing 0 zero-length branches
1936 1586 1573
1586 1936 1573
Processing RSVA F
Removing 0 zero-length branches
1936 1939 1904
1939 1936 1904
Processing RSVA M2-1
Removing 0 zero-length branches
1936 1667 1634
1667 1936 1634
Processing RSVA M2-2
Removing 0 zero-length branches
1936 1951 1916
1951 1936 1916
Processing RSVA L
Removing 0 zero-length branches
1936 1782 1749
1782 1936 1749
Processing RSVB NS1
Removing 0 zero-length branches
1864 1788 1765
1788 1864 1765
Processing RSVB NS2
Removin

In [36]:
#RUN FUBAR FOR PGCOE / RV SAMPLES

targets = ["RSVA", "RSVB"]
genelist = geneids

#outdir = "hyphy_pgcoe"
#samples = "inputs/samples_pgcoe.txt"
dataset = "RV"
#dataset = "PGCOE"


for target in targets:
    outdir = f"hyphy_{dataset}"
    samples = f"inputs/{target}_{dataset}.samples"

    reff = f'refs/{target}.fasta'
    gfff = f'refs/{target}.gff3'
    db = gffutils.create_db(gfff, dbfn=':memory:')
    #run fubar for all genes in genelist
    for gene in genelist:
        print(f"Processing {target} {gene} ({outdir})")

        #codon alignment
        codonf = f"geneseqs/{target}_{gene}_codonaln.fasta"             #nucleotide codon alignment
        codonfsamp = f"geneseqs/{target}_{gene}_codonaln_{dataset}.fasta"   #nucleotide codon alignment

        #parse only samples from pruned tree / sequences
        wgstree = f"geneseqs/{target}_{gene}_tree_pruned.nwk"           #pruned wg tree
        wgstrim = f'geneseqs/{target}_{gene}_tree_pruned_{dataset}.nwk'

        #hyphy outputs
        fubarout = f"hyphy/{target}_{gene}_fubar_{dataset}.json"                  #FUBAR output
        fubarcsv = f"hyphy/{target}_{gene}_fubar_table_{dataset}.csv"                  #FUBAR table output

        #prune tree to sequences, filter sequences to tree
        prune_tree_to_samples(wgstree, samples, wgstrim)
        n = filter_seqs_to_samples(codonf, samples, codonfsamp)
        if n > 5:
            run_fubar(codonfsamp, wgstrim, fubarout)
            parse_fubar_table(fubarout).to_csv(fubarcsv, index=False)
        else:
            print(f"Skipping {target} {gene}, only {n} sequences")


Processing RSVA NS1 (hyphy_RV)
Removing 0 zero-length branches
1692 67 49
1692 67 49
Processing RSVA NS2 (hyphy_RV)
Removing 0 zero-length branches
1682 67 38
1682 67 38
Processing RSVA N (hyphy_RV)
Removing 0 zero-length branches
1404 67 1
1404 67 1
Skipping RSVA N, only 1 sequences
Processing RSVA P (hyphy_RV)
Removing 0 zero-length branches
1767 67 44
1767 67 44
Processing RSVA M (hyphy_RV)
Removing 0 zero-length branches
1899 67 64
1899 67 64
Processing RSVA SH (hyphy_RV)
Removing 0 zero-length branches
1778 67 51
1778 67 51
Processing RSVA G (hyphy_RV)
Removing 0 zero-length branches
1573 67 62
1573 67 62
Processing RSVA F (hyphy_RV)
Removing 0 zero-length branches
1904 67 66
1904 67 66
Processing RSVA M2-1 (hyphy_RV)
Removing 0 zero-length branches
1634 67 32
1634 67 32
Processing RSVA M2-2 (hyphy_RV)
Removing 0 zero-length branches
1916 67 67
1916 67 67
Processing RSVA L (hyphy_RV)
Removing 0 zero-length branches
1749 67 38
1749 67 38
Processing RSVB NS1 (hyphy_RV)
Removing 0 ze

In [ ]:
#run absrel for all genes in genelist
for gene in genelist:
    #codon alignment
    codonf = f"geneseqs/{target}_{gene}_codonaln.fasta"             #nucleotide codon alignment

    #filter tree / sequences to match
    wgstrim = f"geneseqs/{target}_{gene}_tree_pruned.nwk"           #pruned wg tree

    #hyphy outputs
    absrelout = f"hyphy/{target}_{gene}_absrel.json"                  #FUBAR output

    #prune tree to sequences, filter sequences to tree
    #prune_tree_to_seqs(wgstree, codonf, wgstrim)
    #filter_seqs_to_tree(codonf, wgstree, codonf)

    #run_absrel(codonf, wgstrim, absrelout)



target = "RSVA"
genelist = geneids

for gene in genelist:
    fubarjson = f"hyphy/{target}_{gene}_fubar.json"                  #FUBAR output
    fubercsv = f"hyphy/{target}_{gene}_fubar_table.csv"                  #FUBAR table output
    parse_fubar_table(fubarjson).to_csv(fubercsv, index=False)

In [ ]:
#check single gene, N appears to have very low pass rate

target="RSVA"
gene="N"

reffasta = f"refgenes/{target}_{gene}_aa.fasta"                #reference amino acid sequence

#input sequences
gfasta = f"geneseqs/{target}_{gene}_sequences.fasta"           #nucleotide sequences
gaafasta = f"geneseqs/{target}_{gene}_aa.fasta"                #amino acid sequences

#make codon alignment
gmafftf = f"geneseqs/{target}_{gene}_aa_mafft.fasta"            #amino acid alignment
codonf = f"geneseqs/{target}_{gene}_codonaln.fasta"             #nucleotide codon alignment

#filter tree / sequences to match
wgstrim = f"geneseqs/{target}_{gene}_tree_pruned.nwk"           #pruned wg tree

#hyphy outputs
fubarout = f"hyphy/{target}_{gene}_fubar.json"                  #FUBAR output


#extract gene region for all samples, filter sequences with > 5% Ns
gfasta_all = f"geneseqs/{target}_{gene}_sequences_all.fasta"           #nucleotide sequences
gaafasta_all = f"geneseqs/{target}_{gene}_aa_all.fasta"                #amino acid sequences

extract_gene_sequences(fastaf, gfff, gene, gfasta_all, gaafasta_all)


nkeep, ndrop = filter_n_sequences(gfasta_all, gaafasta_all, gfasta, gaafasta, n_threshold=0.05)
print(f"{gene:5s} kept {nkeep:4d}, dropped{ndrop:4d}  (pass:{nkeep / (nkeep + ndrop):6.2%})")